# Étteremértékelések elemzése – 04. notebook
## Google Local review-adatok kiválasztása és előkészítése

**Alkalmazási adat:** Google Local Data (2021) – District of Columbia

Ebben a notebookban a benchmark-adatokról áttérünk a valós alkalmazási adatokra.

**Cél:**
- kb. 20 étterem/kávézó/brunchhely;
- helyenként 50–100 használható angol review;
- review-szintű és mondatszintű adatállomány;
- export a következő, place profile notebook számára.

A review-kat mondatokra bontjuk, mert az Aspect Detection modell mondatszintű SemEval-adatokon tanult.


## 1. Csomagok és beállítások

In [ ]:
!pip -q install langdetect nltk

In [ ]:
import gzip, json, re, urllib.request
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import nltk
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException

DetectorFactory.seed = 42
RANDOM_STATE = 42
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_columns", 30)
print("Környezet kész.")


## 2. UCSD adatforrás

A District of Columbia üzleti metadata-fájlt és a kisebb, class projecthez alkalmasabb **10-core review** fájlt használjuk.


In [ ]:
META_URL = "https://mcauleylab.ucsd.edu/public_datasets/gdrive/googlelocal/meta-District_of_Columbia.json.gz"
REVIEWS_URL = "https://mcauleylab.ucsd.edu/public_datasets/gdrive/googlelocal/review-District_of_Columbia_10.json.gz"

META_PATH = Path("meta-District_of_Columbia.json.gz")
REVIEWS_PATH = Path("review-District_of_Columbia_10.json.gz")

def download_if_missing(url, path):
    if path.exists():
        print("Már létezik:", path)
        return
    print("Letöltés:", path)
    urllib.request.urlretrieve(url, path)
    print(f"Kész: {path} ({path.stat().st_size / 1024**2:.1f} MB)")

download_if_missing(META_URL, META_PATH)
download_if_missing(REVIEWS_URL, REVIEWS_PATH)


## 3. Metadata beolvasása

In [ ]:
def read_jsonl_gz(path):
    rows = []
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    return pd.DataFrame(rows)

meta_df = read_jsonl_gz(META_PATH)
print("Businesses:", len(meta_df))
display(meta_df.head())


## 4. Vendéglátóhely-jelöltek

Restaurant, cafe/coffee, brunch/breakfast, bakery, bistro és diner kategóriákból indulunk. A metadata alapján legalább 80 review-val rendelkező helyeket tartjuk meg.


In [ ]:
CATEGORY_KEYWORDS = [
    "restaurant", "cafe", "coffee", "brunch",
    "breakfast", "bakery", "bistro", "diner"
]
MIN_METADATA_REVIEWS = 80

def normalize_categories(x):
    return [str(v).strip() for v in x] if isinstance(x, list) else []

def category_text(x):
    return " | ".join(normalize_categories(x)).lower()

def is_food_venue(x):
    t = category_text(x)
    return any(k in t for k in CATEGORY_KEYWORDS)

def classify_venue_type(x):
    t = category_text(x)
    if any(k in t for k in ["coffee", "cafe", "café"]):
        return "Cafe/Coffee"
    if any(k in t for k in ["brunch", "breakfast", "bakery"]):
        return "Brunch/Breakfast/Bakery"
    return "Restaurant"

cand = meta_df.loc[meta_df["category"].apply(is_food_venue)].copy()
cand["num_of_reviews"] = pd.to_numeric(cand["num_of_reviews"], errors="coerce")
cand = cand.loc[cand["num_of_reviews"].fillna(0) >= MIN_METADATA_REVIEWS].copy()
cand["venue_type"] = cand["category"].apply(classify_venue_type)
cand["category_text"] = cand["category"].apply(lambda x: " | ".join(normalize_categories(x)))

cols = ["gmap_id","name","address","venue_type","category_text",
        "avg_rating","num_of_reviews","price"]
candidates_df = cand[cols].drop_duplicates("gmap_id").reset_index(drop=True)

print("Jelöltek:", len(candidates_df))
display(candidates_df.sort_values("num_of_reviews", ascending=False).head(20))


## 5. Használható review-k megszámolása a 10-core fájlban

A metadata review-száma nem azonos a 10-core részhalmazban elérhető szöveges review-k számával, ezért ezt külön megszámoljuk.


In [ ]:
candidate_ids = set(candidates_df["gmap_id"].dropna().astype(str))
review_counts = Counter()

with gzip.open(REVIEWS_PATH, "rt", encoding="utf-8") as f:
    for line in f:
        try:
            row = json.loads(line)
        except json.JSONDecodeError:
            continue
        gid = str(row.get("gmap_id", ""))
        if gid not in candidate_ids:
            continue
        text = row.get("text")
        if isinstance(text, str) and len(text.strip().split()) >= 4:
            review_counts[gid] += 1

candidates_df["text_reviews_10core"] = (
    candidates_df["gmap_id"].astype(str).map(review_counts).fillna(0).astype(int)
)

display(candidates_df.sort_values("text_reviews_10core", ascending=False).head(30))


## 6. 20 hely kiválasztása

Tervezett összetétel:
- 10 Restaurant
- 5 Cafe/Coffee
- 5 Brunch/Breakfast/Bakery

Minimum 50 szöveges review szükséges a 10-core részhalmazban. Ha valamelyik típusból nincs elég hely, automatikusan a következő legjobb jelöltekkel töltjük fel a mintát.


In [ ]:
N_PLACES = 20
MIN_TEXT_REVIEWS = 50
QUOTAS = {
    "Restaurant": 10,
    "Cafe/Coffee": 5,
    "Brunch/Breakfast/Bakery": 5
}

eligible = candidates_df.loc[
    candidates_df["text_reviews_10core"] >= MIN_TEXT_REVIEWS
].sort_values(["text_reviews_10core","num_of_reviews"], ascending=False)

parts = []
for venue_type, quota in QUOTAS.items():
    parts.append(eligible.loc[eligible["venue_type"] == venue_type].head(quota))

selected_places = pd.concat(parts, ignore_index=True).drop_duplicates("gmap_id")

if len(selected_places) < N_PLACES:
    missing = N_PLACES - len(selected_places)
    extras = eligible.loc[
        ~eligible["gmap_id"].isin(selected_places["gmap_id"])
    ].head(missing)
    selected_places = pd.concat([selected_places, extras], ignore_index=True)

selected_places = selected_places.head(N_PLACES).reset_index(drop=True)
selected_places.insert(0, "place_index", range(1, len(selected_places)+1))

print("Kiválasztott helyek:", len(selected_places))
display(selected_places[[
    "place_index","name","venue_type","avg_rating",
    "num_of_reviews","text_reviews_10core","category_text"
]])


### Ellenőrzési pont

Itt nézd át röviden a 20 helyet. A kiválasztás reprodukálható szabályok alapján történik, nem kézi cherry-pickinggel.


## 7. Review-k begyűjtése

Első körben helyenként legfeljebb 130 review-t gyűjtünk, hogy a nyelvi szűrés után is lehetőleg maradjon 100.


In [ ]:
COLLECT_PER_PLACE = 130
selected_ids = set(selected_places["gmap_id"].astype(str))
reviews_by_place = {gid: [] for gid in selected_ids}

with gzip.open(REVIEWS_PATH, "rt", encoding="utf-8") as f:
    for line in f:
        try:
            row = json.loads(line)
        except json.JSONDecodeError:
            continue

        gid = str(row.get("gmap_id", ""))
        if gid not in selected_ids or len(reviews_by_place[gid]) >= COLLECT_PER_PLACE:
            continue

        text = row.get("text")
        if not isinstance(text, str):
            continue

        text = re.sub(r"\s+", " ", text).strip()
        if len(text.split()) < 4:
            continue

        reviews_by_place[gid].append({
            "gmap_id": gid,
            "user_id": row.get("user_id"),
            "time": row.get("time"),
            "rating": row.get("rating"),
            "review_text": text
        })

reviews_df = pd.DataFrame(
    [r for rows in reviews_by_place.values() for r in rows]
)

print("Begyűjtött review-k:", len(reviews_df))
display(reviews_df.head())


## 8. Angol nyelvű review-k

In [ ]:
def detect_language(text):
    try:
        return detect(text)
    except LangDetectException:
        return None

reviews_df["language"] = reviews_df["review_text"].apply(detect_language)
print(reviews_df["language"].value_counts(dropna=False).head(10))

english_reviews = reviews_df.loc[reviews_df["language"] == "en"].copy()
print("Angol review-k:", len(english_reviews))


## 9. Végleges review-minta

Helyenként maximum 100 angol review-t választunk, reprodukálható random samplinggel.


In [ ]:
MAX_REVIEWS_PER_PLACE = 100
parts = []

for gid, group in english_reviews.groupby("gmap_id"):
    parts.append(
        group.sample(
            n=min(MAX_REVIEWS_PER_PLACE, len(group)),
            random_state=RANDOM_STATE
        )
    )

selected_reviews = pd.concat(parts, ignore_index=True)

place_info = selected_places[[
    "place_index","gmap_id","name","venue_type",
    "category_text","avg_rating","price"
]]

selected_reviews = selected_reviews.merge(place_info, on="gmap_id", how="left")
selected_reviews["review_id"] = (
    selected_reviews["gmap_id"].astype(str) + "_" +
    selected_reviews["user_id"].astype(str) + "_" +
    selected_reviews["time"].astype(str)
)

review_summary = (
    selected_reviews
    .groupby(["place_index","name","venue_type"], as_index=False)
    .agg(
        selected_reviews=("review_id","count"),
        mean_review_rating=("rating","mean")
    )
    .sort_values("place_index")
)

display(review_summary)
print("Összes végleges review:", len(selected_reviews))


## 10. Review-k mondatokra bontása

A modellezés inputja mondat lesz, nem teljes review.


In [ ]:
try:
    nltk.download("punkt", quiet=True)
    nltk.download("punkt_tab", quiet=True)
except Exception:
    pass

def split_sentences(text):
    try:
        sentences = nltk.sent_tokenize(text)
    except Exception:
        sentences = re.split(r"(?<=[.!?])\s+", text)

    result = []
    for s in sentences:
        s = re.sub(r"\s+", " ", s).strip()
        if len(s.split()) >= 3 and len(s) <= 600:
            result.append(s)
    return result

rows = []
for _, row in selected_reviews.iterrows():
    for i, sentence in enumerate(split_sentences(row["review_text"]), start=1):
        rows.append({
            "place_index": row["place_index"],
            "gmap_id": row["gmap_id"],
            "name": row["name"],
            "venue_type": row["venue_type"],
            "review_id": row["review_id"],
            "review_rating": row["rating"],
            "sentence_index": i,
            "sentence": sentence
        })

selected_sentences = pd.DataFrame(rows)
selected_sentences["sentence_id"] = (
    selected_sentences["review_id"] + "_s" +
    selected_sentences["sentence_index"].astype(str)
)

print("Review-k:", len(selected_reviews))
print("Mondatok:", len(selected_sentences))
print("Átlag mondat/review:", round(len(selected_sentences)/len(selected_reviews), 2))
display(selected_sentences.head(15))


## 11. Gyors adatminőség-ellenőrzés

In [ ]:
quality_summary = (
    selected_sentences
    .groupby(["place_index","name","venue_type"], as_index=False)
    .agg(
        reviews=("review_id","nunique"),
        sentences=("sentence_id","count"),
        mean_sentence_words=("sentence", lambda s: np.mean([len(x.split()) for x in s]))
    )
)

display(quality_summary)
assert selected_sentences["sentence"].notna().all()
assert (selected_sentences["sentence"].str.len() > 0).all()
print("Alap ellenőrzések rendben.")


## 12. Export

Három fájl készül:
- `selected_places.csv`
- `selected_reviews.csv`
- `selected_review_sentences.csv`

A következő notebookban elsősorban a mondatszintű fájlt használjuk.


In [ ]:
PLACES_PATH = Path("selected_places.csv")
REVIEWS_OUT_PATH = Path("selected_reviews.csv")
SENTENCES_PATH = Path("selected_review_sentences.csv")

selected_places.to_csv(PLACES_PATH, index=False)
selected_reviews.to_csv(REVIEWS_OUT_PATH, index=False)
selected_sentences.to_csv(SENTENCES_PATH, index=False)

print("Elmentve:")
print("-", PLACES_PATH)
print("-", REVIEWS_OUT_PATH)
print("-", SENTENCES_PATH)


## 13. CSV-k letöltése Colabból

Ezt a lépést mindenképpen futtasd le: a Colab runtime tartalma nem tartós.


In [ ]:
try:
    from google.colab import files
    files.download(str(PLACES_PATH))
    files.download(str(REVIEWS_OUT_PATH))
    files.download(str(SENTENCES_PATH))
except ImportError:
    print("A CSV-k a notebook mappájában találhatók.")


# 14. Következő lépés

A következő notebookban:

**Google review sentence → Aspect Detection → Sentiment Classification → helyenkénti aggregálás → Place Profile**

Ezek a place profile-ok lesznek a személyre szabott recommendation scoring alapjai.
